# 🗄️ Construcción e Indexación de Memoria Vectorial (RAG) en Kaggle
### Indexación del Corpus TSQA con ChromaDB y SentenceTransformers (all-mpnet-base-v2)

Este cuaderno construye la base de datos vectorial persistente utilizada por el **Agente Maestro** para la recuperación de semillas semánticas y temporales en el pipeline de la tesis.

## 1. Instalación de Dependencias

In [ ]:
!pip install -q chromadb sentence-transformers psutil

## 2. Detección de Dispositivo y GPU

In [ ]:
import torch
import psutil

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo activo: {device}")
if device == "cuda":
    print(f"GPU detectada: {torch.cuda.get_device_name(0)}")
    print(f"VRAM disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


## 3. Localización Automática del Dataset TSQA

In [ ]:
import os
from pathlib import Path

# Búsqueda automática en /kaggle/input o directorio local
posibles_rutas = [
    "/kaggle/input/ts-dataset-jsonl/TS_Dataset.jsonl",
    "/kaggle/input/tsqa-time-series/time_series_generation.jsonl",
    "/kaggle/input/datasets/danielrangelsanch/ts-dataset-jsonl/TS_Dataset.jsonl",
    "./data/time_series_generation.jsonl",
    "./data/TS_Dataset.jsonl",
    "time_series_generation.jsonl",
    "TS_Dataset.jsonl"
]

jsonl_path = None
for r in posibles_rutas:
    if os.path.exists(r):
        jsonl_path = r
        break

# Búsqueda recursiva en /kaggle/input si no se halló directamente
if jsonl_path is None and os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f.endswith(".jsonl") and ("ts" in f.lower() or "time_series" in f.lower()):
                jsonl_path = os.path.join(root, f)
                break
        if jsonl_path:
            break

if jsonl_path is None:
    jsonl_path = "time_series_generation.jsonl"
    print(f"⚠️ No se encontró dataset precargado. Asumiendo archivo local: {jsonl_path}")
else:
    print(f"✅ Dataset TSQA localizado en: {jsonl_path}")

CHROMA_DB_PATH = "/kaggle/working/rag_maestro" if os.path.exists("/kaggle/working") else "./rag/maestro"
COLLECTION_NAME = "time_series_generation"
print(f"📁 Directorio destino de ChromaDB: {CHROMA_DB_PATH}")


## 4. Carga de Modelo y Funciones de Extracción

In [ ]:
import json
import re
import ast
from sentence_transformers import SentenceTransformer
import chromadb

PATTERN_LISTA = re.compile(
    r'\[\s*(?:-?\d+(?:\.\d+)?|'[\dA-Za-z.-]+'|"[\dA-Za-z.-]+")'
    r'(?:\s*,\s*(?:-?\d+(?:\.\d+)?|'[\dA-Za-z.-]+'|"[\dA-Za-z.-]+"))*\s*\]'
)

def extract_all_time_series(text):
    """Extrae listas numéricas utilizando ast.literal_eval."""
    series = []
    for match in PATTERN_LISTA.finditer(text):
        try:
            lst = ast.literal_eval(match.group())
            if isinstance(lst, list):
                series.append(lst)
        except (SyntaxError, ValueError):
            pass
    return series

def clean_description(text):
    """Limpia los números largos del texto para indexar solo el contexto semántico."""
    return re.sub(r"\s+", " ", PATTERN_LISTA.sub("", text)).strip()

def build_search_text(entry):
    domain = entry.get("metadata", {}).get("original_domain", "General")
    task = entry.get("metadata", {}).get("task_type", "forecasting")
    question = clean_description(entry.get("input", ""))
    response = entry.get("output", "")
    return f"Dominio: {domain}. Tarea: {task}. Pregunta: {question}. Respuesta: {response}."

print("⏳ Cargando SentenceTransformer (all-mpnet-base-v2)...")
model = SentenceTransformer("all-mpnet-base-v2", device=device)
print("✅ Modelo de embeddings listo.")


## 5. Indexación por Lotes en ChromaDB

In [ ]:
import os

os.makedirs(CHROMA_DB_PATH, exist_ok=True)
client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

try:
    client.delete_collection(COLLECTION_NAME)
    print(f"Colección previa '{COLLECTION_NAME}' eliminada.")
except Exception:
    pass

collection = client.create_collection(name=COLLECTION_NAME)

EMBEDDING_BATCH_SIZE = 64 if device == "cuda" else 16
print(f"Tamaño de lote para embeddings: {EMBEDDING_BATCH_SIZE}")

lines_buffer = []
entries_buffer = []
total_indexed = 0

def flush_batch(ids, embs, metas, docs):
    if ids:
        collection.add(
            ids=ids,
            embeddings=embs,
            metadatas=metas,
            documents=docs
        )

print(f"🚀 Procesando dataset: {jsonl_path}...")
with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        line_str = line.strip()
        if not line_str:
            continue
        entry = json.loads(line_str)

        series_input = extract_all_time_series(entry.get("input", ""))
        series_output = extract_all_time_series(entry.get("output", ""))
        series_data = {
            "input": [{"longitud_puntos": len(x), "serie": x} for x in series_input],
            "output": [{"longitud_puntos": len(x), "serie": x} for x in series_output]
        }
        series_json = json.dumps(series_data, separators=(",", ":"))
        search_text = build_search_text(entry)
        doc_id = str(entry.get("id", total_indexed + len(entries_buffer)))

        lines_buffer.append(search_text)
        entries_buffer.append({
            "id": doc_id,
            "metadata": {
                "id": doc_id,
                "domain": str(entry.get("metadata", {}).get("original_domain", "General")),
                "task_type": str(entry.get("metadata", {}).get("task_type", "forecasting")),
                "question": str(entry.get("input", "")),
                "response": str(entry.get("output", "")),
                "time_series": series_json
            }
        })

        if len(lines_buffer) >= EMBEDDING_BATCH_SIZE:
            embs = model.encode(lines_buffer, batch_size=EMBEDDING_BATCH_SIZE, show_progress_bar=False)
            ids_b = [e["id"] for e in entries_buffer]
            metas_b = [e["metadata"] for e in entries_buffer]
            flush_batch(ids_b, embs.tolist(), metas_b, lines_buffer)
            total_indexed += len(ids_b)
            if total_indexed % 500 == 0:
                print(f"  -> Indexados {total_indexed} documentos...", end="")
            lines_buffer.clear()
            entries_buffer.clear()

if lines_buffer:
    embs = model.encode(lines_buffer, batch_size=EMBEDDING_BATCH_SIZE, show_progress_bar=False)
    ids_b = [e["id"] for e in entries_buffer]
    metas_b = [e["metadata"] for e in entries_buffer]
    flush_batch(ids_b, embs.tolist(), metas_b, lines_buffer)
    total_indexed += len(ids_b)
    lines_buffer.clear()
    entries_buffer.clear()

print(f"
🎉 ¡Indexación completada!")
print(f"Total documentos indexados en ChromaDB: {collection.count()}")


## 6. Verificación de Búsqueda Semántica

In [ ]:
consulta_prueba = "Daily temperature variations with seasonal patterns and sudden weather events"
emb_q = model.encode(consulta_prueba).tolist()
resultados = collection.query(query_embeddings=[emb_q], n_results=3)

print(f"🔍 Consulta de prueba: '{consulta_prueba}'
")
for i, (doc_id, meta, dist) in enumerate(zip(resultados["ids"][0], resultados["metadatas"][0], resultados["distances"][0])):
    print(f"[{i+1}] ID: {doc_id} | Dominio: {meta['domain']} | Distancia: {dist:.4f}")
    print(f"    Pregunta: {meta['question'][:120]}...")
    print()


## 7. Comprimir la Base Vectorial para Descarga o Dataset de Kaggle

In [ ]:
import shutil

zip_output = "/kaggle/working/rag_maestro" if os.path.exists("/kaggle/working") else "./rag_maestro"
print(f"📦 Comprimiendo base de datos ChromaDB en '{zip_output}.zip'...")
shutil.make_archive(zip_output, "zip", CHROMA_DB_PATH)
print(f"✅ Archivo '{zip_output}.zip' generado exitosamente.")
